# Fruit Classification — 01 · Exploratory Data Analysis

This project builds and compares two image classifiers — a small custom CNN and a transfer-learning model based on **VGG16** — on a 24-class subset of the [Fruits-360 dataset](https://www.kaggle.com/datasets/moltean/fruits) (original-size version, 2021.09.12.0).

The work is split into four notebooks:

| Notebook | Purpose |
|---|---|
| **01_eda** (this one) | Understand the data: class distribution, image sizes, and a closer look at how the official train/val/test split was created |
| **02_preprocessing** | Build a leakage-aware train/val/test split (`data/splits.csv`) |
| **03_modelling_keras** | Small CNN + VGG16 with TensorFlow/Keras |
| **04_modelling_pytorch** | Custom CNN + VGG16 with PyTorch |

**Key question for this notebook:** can we trust the official train/validation/test split, or does it leak information? (Spoiler: it leaks — see section 5.)

## 1. Setup

The data lives inside this repository under `data/Images/`. When running on **Google Colab**, the cell below clones the repository first; when running locally, it simply uses the relative path.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("cnn_vgg16__fruit_classification"):
        !git clone https://github.com/Adriana394/cnn_vgg16__fruit_classification.git
    REPO_DIR = "cnn_vgg16__fruit_classification"
else:
    REPO_DIR = ".."  # this notebook lives in <repo>/notebooks/

DATA_DIR = os.path.join(REPO_DIR, "data", "Images")
print("Data directory:", os.path.abspath(DATA_DIR))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

## 2. Dataset overview

The dataset ships with three folders — `Training`, `Validation` and `Test` — each containing one subfolder per class.

Every image file is named `r<axis>_<frame>.jpg`, e.g. `r0_103.jpg`:

- **axis** (`r0`, `r1`, `r2`): each fruit was placed on a slowly rotating motor and filmed; some fruits were filmed around more than one rotation axis, and each axis is a separate video.
- **frame**: the frame index within that video. Consecutive frames are taken a fraction of a second apart, so they look almost identical.

We first index every image into a single DataFrame, parsing the axis and frame number from the file name.

In [ ]:
records = []
for split in ["Training", "Validation", "Test"]:
    split_dir = os.path.join(DATA_DIR, split)
    for label in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for filename in sorted(os.listdir(class_dir)):
            axis, frame = filename.rsplit(".", 1)[0].rsplit("_", 1)
            records.append(
                {
                    "split": split,
                    "label": label,
                    "axis": axis,
                    "frame": int(frame),
                    "filepath": os.path.join(split_dir, label, filename),
                }
            )

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
print(f"Number of classes: {df['label'].nunique()}")
df.head()

In [ ]:
split_sizes = df["split"].value_counts().reindex(["Training", "Validation", "Test"])
split_sizes.to_frame("images").assign(share=lambda d: (d["images"] / len(df)).round(3))

### Class distribution

How many images does each class have, and is the split ratio consistent across classes?

In [ ]:
class_counts = (
    df.pivot_table(index="label", columns="split", values="filepath", aggfunc="count")
    .reindex(columns=["Training", "Validation", "Test"])
    .sort_values("Training", ascending=False)
)

class_counts.plot.bar(stacked=True, figsize=(14, 5), width=0.8)
plt.title("Images per class and split")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.legend(title="Split")
plt.tight_layout()
plt.show()

class_counts.assign(total=class_counts.sum(axis=1))

The classes are reasonably balanced (no class is orders of magnitude larger than another), and the ~50/25/25 split ratio is consistent across classes. Classes filmed around several rotation axes (e.g. the apples with `r0` and `r1`) have roughly twice as many images as single-axis classes.

Plain accuracy is therefore an acceptable headline metric, but we will still look at per-class results (confusion matrix, classification report) in the modelling notebooks.

## 3. Image sizes

Unlike the well-known 100×100 version of Fruits-360, this dataset version contains images at their **original filmed size**, so sizes can vary. Let's check a sample.

In [ ]:
sample = df.sample(n=500, random_state=42)
sizes = sample["filepath"].map(lambda p: Image.open(p).size)

size_counts = sizes.value_counts()
print(f"Distinct (width, height) values in the sample: {len(size_counts)}")
size_counts.head(10)

The images come in many different sizes, so the input pipeline of both models must **resize every image** to a fixed shape. We will use **100×100 pixels** throughout the project — small enough to train quickly, and consistent with the classic Fruits-360 setup.

## 4. Example images

One training example per class. All photos show a single fruit/vegetable in front of a white background, extracted by the dataset authors with a flood-fill algorithm — so the background carries almost no information and the task is mostly about shape, colour and texture.

In [ ]:
train_df = df[df["split"] == "Training"]

fig, axes = plt.subplots(4, 6, figsize=(15, 11))
for ax, (label, group) in zip(axes.ravel(), train_df.groupby("label")):
    image = Image.open(group.iloc[0]["filepath"]).resize((100, 100))
    ax.imshow(image)
    ax.set_title(label, fontsize=9)
    ax.axis("off")

plt.suptitle("One example per class (Training split)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Data leakage in the official split

According to the dataset README, the official split was created **per object and frame index** with this rule (`k` is a multiple of 4):

- frames `k` and `k + 2` → **Training**
- frame `k + 1` → **Validation**
- frame `k + 3` → **Test**

That means the three splits *interleave frames of the same video*: for almost every validation/test image there is a training image showing **the same physical fruit, in almost the same rotation, a fraction of a second apart**. A model can score very high on validation/test simply by memorising each fruit — this is **data leakage**, and the resulting metrics say little about generalisation.

Let's verify the interleaving pattern on one video (class `apple_6`, rotation axis `r0`):

In [ ]:
video = (
    df[(df["label"] == "apple_6") & (df["axis"] == "r0")]
    .sort_values("frame")
    .reset_index(drop=True)
)
video[["frame", "split"]].head(12)

Exactly the documented pattern. Now the visual proof — three **consecutive frames** of the same video, which the official split sends to three different sets:

In [ ]:
neighbours = video[video["frame"].isin([0, 1, 3])]

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, (_, row) in zip(axes, neighbours.iterrows()):
    ax.imshow(Image.open(row["filepath"]))
    ax.set_title(f"frame {row['frame']} → {row['split']}")
    ax.axis("off")

plt.suptitle("Near-identical neighbouring frames end up in different splits", fontsize=13)
plt.tight_layout()
plt.show()

These images are practically indistinguishable, yet one is used for training and the others for validation/testing. Any evaluation on this split is heavily inflated.

**Consequence:** in `02_preprocessing` we replace the official split with a **contiguous-block split**: for each class and rotation axis, the frames are sorted by index and cut into one consecutive block per set (≈70% train / 15% validation / 15% test). Then only the few frames at the two block boundaries are temporal neighbours across sets, instead of *every* frame.

One honest limitation remains: each class consists of only **one physical object** (the class names even carry the object number, e.g. `apple_golden_1` vs. `apple_golden_2`). A truly leakage-free split would need *different objects* in train and test, which this dataset cannot provide. The block split removes the near-duplicate-frame leakage, but the models are still evaluated on unseen *views*, not unseen *fruits* — we will keep that in mind when interpreting the results.

## 6. Findings

1. **12,455 images, 24 classes** (apples, pears, cucumbers, zucchini, carrot, cabbage, eggplant), split ~50/25/25 into Training/Validation/Test.
2. **Classes are reasonably balanced**; multi-axis classes have about twice the images of single-axis classes. Accuracy is usable as the main metric, complemented by per-class reports.
3. **Image sizes vary** (original filmed size) → all pipelines resize to 100×100.
4. **The official split leaks**: consecutive, near-identical video frames are interleaved across Training/Validation/Test. We replace it with a contiguous-block split per class and rotation axis in `02_preprocessing`.
5. **Remaining limitation**: one physical object per class, so test images show known fruits from unseen angles. Reported metrics measure view generalisation, not object generalisation.